# 07 · Damping test: does lowering θ_L reproduce the natural amplitude loss?

Within a recording the rhythm damps as cells drift out of synchrony; period and phase are kept while amplitude shrinks. If θ is a biologically meaningful amplitude control, the early window's z with a lower θ_L should resemble the window one cycle later.

Design (fixed in `src/analysis/damping_test.py` before results are seen):
- Traces: MSB recordings excluding U-2 OS. Early window: first start ≥ 12 h. Late window: start closest to early + 1 cosinor period (same phase), mismatch ≤ 2 h.
- Models of the late window, each from the early window only: unchanged reconstruction, reconstruction × best scalar, early z with best θ_L (grid 0.02–0.98), and the late window's own reconstruction (ceiling).
- H1 amplitude ratio late/early < 1; H2 θ* below the inferred θ_L when damping; H3 θ beats the unchanged reconstruction; H4 shape correlation is best at a θ_L below the inferred one. θ vs plain scaling is reported without a directional prior.
- Known from the inputs: after one cycle the amplitude is ≈ 0.55× (median), beyond the ≈ 0.8× lower limit of θ in the s1 models, so θ* is expected to hit the lower bound often; `coverage` measures how much of the damping θ reproduces.

Runs on CPU. Outputs: `results/damping/<run>/`.

In [ ]:
import os, sys
from pathlib import Path
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
sys.dont_write_bytecode = True
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run.py").exists() and (p / "configs").exists())
sys.path.insert(0, str(ROOT))
import pandas as pd
from src.analysis import damping_test as dmp
pd.set_option("display.width", 200, "display.max_columns", 40)
%matplotlib inline

RUN = "s1/s1_MULTI_FC12_Z10_s0"   # stage/run_id of a finished sweep run
WHICH = "all"                      # "all" (train + validation) or "val"

## 1. Pairs and fits

In [ ]:
df, ex = dmp.run(RUN, which=WHICH)
summary = dmp.summarise(df)
out = dmp.save(RUN, df, summary)
print("saved:", out)
pd.Series(summary).round(3)

## 2. Overview

In [ ]:
fig = dmp.plot_summary(df)
fig.savefig(out / "summary.png", dpi=150, bbox_inches="tight")

## 3. Examples (damping traces spanning the observed range)

In [ ]:
fig = dmp.plot_examples(df, ex)
fig.savefig(out / "examples.png", dpi=150, bbox_inches="tight")

## 4. Per trace

In [ ]:
cols = ["trace_id", "cell_line", "reporter", "split", "period_h", "amp_ratio_obs", "amp_ratio_theta", "coverage",
        "theta_l0", "theta_star", "theta_at_bound", "mse_unchanged", "mse_scaled", "mse_theta", "mse_ceiling", "delta_corr"]
df[cols].sort_values("amp_ratio_obs").round(3)

In [ ]:
df.groupby("cell_line")[["amp_ratio_obs", "coverage", "delta_corr"]].median().round(2)